# 07 — Real space vs. redshift space

**Exploratory twin of `scripts/plot_redshift_space_comparison.py`.** Every code cell below
calls one stage function of `dvcorr.pipeline.redshift_space_comparison` (plus the stages it
reuses unchanged from `dvcorr.pipeline.velocity_centered`) in sequence — nothing here is
reimplemented; this notebook and the script drive the identical library functions
(CLAUDE.md's "one library, two thin consumers" model, `notebooks/05_velocity_centered_dipole
.ipynb` / `06_velocity_frame_comparison.ipynb` are the models this one follows).

**What this compares.** The velocity-centered ζ₁ dipole estimator
(`dvcorr.estimators.shell_dipole.velocity_centered_shell_dipole`) is run TWICE, unchanged,
on a SHARED set of centers: once on real (comoving box) positions, once on redshift-space
positions — every halo displaced along its own observer line of sight by its own radial
peculiar velocity, `dvcorr.redshift_space.to_redshift_space`. MDPL2 snapshot 125 is z ≈ 0,
so in the box's comoving coordinates this displacement is EXACT, not an approximation to a
smaller effect — there is no redshift-to-distance conversion anywhere in this pipeline (that
question belongs to the future real-data arm).

**The one surprising fact worth remembering before the plots below:** the displacement is
purely radial, so it does **not** rotate n̂ (verified numerically to ~3.3e-16) — and since
u_α = v_α · n̂_V,α depends only on n̂ and v (both invariant), u_α is **identical** between the
two runs. What changes is SHELL COMPOSITION only — which tracers land in which radial bin
around a given center — never a center's own axis or scalar weight. See
`dvcorr.redshift_space`'s module docstring for the full argument.

In [ ]:
%matplotlib inline

import numpy as np

from dvcorr import conventions
from dvcorr.config import PathsConfig
from dvcorr.pipeline.velocity_centered import draw_candidates
from dvcorr.pipeline.redshift_space_comparison import (
    RedshiftSpaceRunConfig,
    load_and_carve_buffered,
    build_tracer_spaces,
    select_redshift_shared_centers,
    run_both_spaces,
    normalize_redshift_comparison,
    membership_diagnostics,
    make_redshift_comparison_figure,
    make_single_center_figure,
)

## Parameters

`RedshiftSpaceRunConfig` extends `RunConfig` (the same base class
`ComparisonRunConfig` in notebook 06 extends) with the knobs this comparison needs:
`v_margin_statistic` ("max", the default — a TRUE bound on radial displacement, vs.
"percentile" — cheaper but requires an extra global |v_r| cut on centers, see the pipeline
module's docstring), `flip_guard_floor` (the through-observer guard's floor,
`dvcorr.redshift_space.FLIP_GUARD_FLOOR`), `redshift_shuffle_seed` (the redshift-space run's
own null, kept distinct from `shuffle_seed`), and `example_center_index` (which single center
the single-center figure below plots).

In [ ]:
cfg = RedshiftSpaceRunConfig()
paths = PathsConfig()
observer = np.asarray(conventions.OBSERVER_POSITION, dtype=float)

print(cfg)

## 1 — Buffered load: two carves, one catalog read

Tracers displace too, so a plain `R_sub` carve leaves the redshift-space tracer field
incomplete at the boundary in BOTH directions (some real members displace out, some
just-outside halos would have displaced in) — the exact mechanism
`dvcorr.estimators.shell_dipole.core_center_mask`'s docstring blames for the first MDPL2
run's ~+13 km/s null offset, here applied to the tracer field. The fix: carve at
`R_sub + v_margin` first (`load_and_carve_buffered`), where `v_margin` is derived from the
PLAIN `R_sub` population's |v_r| — computed here, once, and threaded through every later
stage rather than recomputed inline.

In [ ]:
buffer = load_and_carve_buffered(cfg, paths)

print(f"v_margin = {buffer.v_margin_kms:.1f} km/s = {buffer.v_margin_mpc:.2f} h^-1 Mpc")
print(f"plain carve: {buffer.n_core} halos; buffered carve: {buffer.n_buffer} halos "
      f"({100.0 * (buffer.n_buffer / buffer.n_core - 1.0):.1f}% more)")

## 2 — Build the tracer fields in both spaces

Displace the FULL buffered tracer catalog, then restrict the result back down to `R_sub` —
restricting first would throw away exactly the boundary tracers the buffer exists to keep.
The flux check below is the trust diagnostic for this step: with the buffer capturing inflow
and outflow both, the real-inside-R_sub and redshift-inside-R_sub counts should agree to
well under a percent. A much larger gap would mean the buffer (`v_margin`) is misconfigured,
not that RSD moved a physically large fraction of the tracer mass.

In [ ]:
tracers = build_tracer_spaces(cfg, buffer, observer)

print(f"real-inside-R_sub = {tracers.n_real_inside}, "
      f"redshift-inside-R_sub = {tracers.n_redshift_inside}")

## 3 — Draw candidate centers

A seeded subsample of the PLAIN-radius carve (`buffer.pos_core` / `buffer.vel_core`) — a
candidate center must itself be a real member of the analysis sub-volume, not merely of the
wider tracer buffer. Identical stage to notebooks 05/06 — `draw_candidates` is imported from
`dvcorr.pipeline.velocity_centered` unchanged.

In [ ]:
s_candidates, v_candidates = draw_candidates(cfg, buffer.pos_core, buffer.vel_core)
print("candidate centers:", s_candidates.shape[0])

## 4 — Select the SHARED center set

This is the cell that makes the comparison a fair one. Three sequential cuts:

1. Core cut + speed floor on REAL positions, with a WIDENED margin `r_max + v_margin` (never
   `r_max` alone, and never checked against a center's DISPLACED position — see the pipeline
   module's docstring for why selecting on the displaced position would be a real bias, not
   just an inelegance: it would systematically drop outward-movers near the far boundary
   while keeping inward-movers, skewing ⟨u⟩ with position).
2. (Only when `v_margin_statistic == "percentile"`) a GLOBAL |v_r| cut — legitimate where the
   displaced-position cut is not, because it is sign-symmetric and position-independent.
3. The through-observer flip guard on each survivor's own displacement — a genuine safety
   net, but the widened real-position margin from step 1 already GUARANTEES (by a
   triangle-inequality argument, spelled out in `select_redshift_shared_centers`'s docstring)
   that every survivor's displaced position clears the plain `r_max` margin the estimator
   re-checks internally, so this step is expected to drop nothing in the ordinary case.

Cost of this fairness: with the default `v_margin` ("max"), the effective core radius shrinks
from `R_sub - r_max` (240, the single-frame run's own margin) to `R_sub - (r_max + v_margin)`
(~210) — a candidate volume fraction of ~34% instead of ~51%, roughly 33% fewer centers than
the currently published single-frame run. That is the price of a like-for-like comparison.

In [ ]:
centers = select_redshift_shared_centers(
    cfg, s_candidates, v_candidates, observer, buffer.v_margin_kms, buffer.v_margin_mpc
)

print()
print(f"n_centers = {centers.n_centers} (shared by both runs)")

## 5 — Run both spaces

`velocity_centered_shell_dipole` is called TWICE, completely unchanged — once on
`centers.s_centers_real` / `tracers.s_tracers_real`, once on `centers.s_centers_redshift` /
`tracers.s_tracers_redshift`, with the SAME `v_centers`, `shell_edges`, `sub_volume_radius`,
and `observer`. `run_both_spaces` asserts both calls report `n_centers ==
centers.n_centers` — the row-alignment invariant this whole comparison depends on.

In [ ]:
results = run_both_spaces(cfg, centers, tracers, observer)

real_result, redshift_result = results.real_result, results.redshift_result
print(f"real:     n_centers={real_result.n_centers}, pair_count.sum()={real_result.pair_count.sum():.0f}")
print(f"redshift: n_centers={redshift_result.n_centers}, pair_count.sum()={redshift_result.pair_count.sum():.0f}")

## 6 — Normalize (shared, real-space n̄)

Both runs are normalized with the SAME n̄, computed from the REAL-space tracer count inside
`R_sub` (`tracers.n_real_inside`) — never the buffered count (that substitution would inflate
n̄ by ~(R_sub + v_margin)³/R_sub³, silently suppressing ζ̂₁ by the same fraction in BOTH runs
with no shape change to reveal it). A per-run n̄ would instead fold the small real-vs-redshift
boundary-flux difference into the amplitude ratio, which is the headline quantity here — so
one shared n̄ it is.

In [ ]:
comparison = normalize_redshift_comparison(cfg, results)

print(f"n_bar = {comparison.n_bar:.3e} tracers / (h^-1 Mpc)^3")
print(f"n_real_inside_r_sub = {comparison.n_real_inside_r_sub}, "
      f"n_redshift_inside_r_sub = {comparison.n_redshift_inside_r_sub}")

## 7 — The comparison figure

Two panels (CLAUDE.md hard rule 6: the dipole never travels without its monopole). **Top:**
ζ̂₁ for both spaces, each with its own across-center SEM band and its own velocity-shuffle
null (dashed, lighter tint) — real space's null uses `cfg.shuffle_seed`, redshift space's
uses `cfg.redshift_shuffle_seed`, distinct streams. **Bottom:** both monopoles, on a SINGLE
shared y-axis this time (unlike notebook 06's twin-axis comparison) — both are the same kind
of quantity, Σ|u_α| N_α,b, on the SAME u_α values (n̂ invariance), differing only through
shell occupancy, so there is no projection-factor scale gap to hide behind separate axes. Any
trend DIFFERENCE between the two monopole curves is coming from occupancy alone — the
finite-distance / incomplete-shell trust diagnostic for this comparison specifically.

In [ ]:
fig = make_redshift_comparison_figure(cfg, results, comparison)

paths.ensure_output_dir()
out_path = paths.output_dir / cfg.comparison_output_name
fig.savefig(out_path, dpi=cfg.dpi, bbox_inches="tight")
print(f"wrote {out_path}")
fig

## 8 — Single-center example

The same two-panel layout, for ONE center (`cfg.example_center_index`, a config parameter —
not hardcoded here), real and redshift space overlaid, with NO error band (a single center
has no across-center population to form one from). The dipole curve uses the same per-shell
normalization as the stacked figure (without the `/n_centers` averaging, since there is only
one center); the monopole is the RAW `per_center_speed[a] * per_center_count[a]` — no n̄·V_b
division, since the point is this one center's own realized shell-by-shell shape, not a
comparison to the ensemble expectation.

In [ ]:
fig_single = make_single_center_figure(cfg, results, comparison.n_bar)

single_center_path = paths.output_dir / cfg.single_center_output_name
fig_single.savefig(single_center_path, dpi=cfg.dpi, bbox_inches="tight")
print(f"wrote {single_center_path}")
fig_single

## 9 — Membership diagnostics

Standalone diagnostic (`membership_diagnostics`), deliberately NOT part of the estimator: it
runs its own `cKDTree` queries against both tracer fields, around the shared centers' REAL
positions, and compares shell membership by STABLE buffer-row id (never by positional array
index — `s_tracers_real` and `s_tracers_redshift` are different subsets of the same buffer,
so row *i* of one and row *i* of the other are, in general, different halos).

- **Net change** (`net_change`): Σ_α (N_redshift,α,b − N_real,α,b) per shell — can be small
  even when membership fully churned underneath.
- **Churn**: `churn_only_real` / `churn_only_redshift` / `churn_intersection` — the count of
  (center, tracer) pairs present in one space but not the other, and in both. A shell with
  `net_change ≈ 0` but large churn on both sides is exactly the case net change alone would
  hide.

In [ ]:
diagnostics = membership_diagnostics(
    centers.s_centers_real,
    tracers.s_tracers_real, tracers.real_ids,
    tracers.s_tracers_redshift, tracers.redshift_ids,
    cfg.shells.shell_edges,
)

for b, r in enumerate(diagnostics.shell_centers):
    print(
        f"r={r:6.1f}  net_change={diagnostics.net_change[b]:+7.1f}  "
        f"only_real={diagnostics.churn_only_real[b]:7.0f}  "
        f"only_redshift={diagnostics.churn_only_redshift[b]:7.0f}  "
        f"intersection={diagnostics.churn_intersection[b]:7.0f}"
    )

## 10 — Summary numbers

A compact read-out: the shell-averaged amplitude in each space, their ratio (the headline
number this whole comparison exists to produce), and the boundary-flux / margin bookkeeping
that says whether the comparison is trustworthy.

In [ ]:
mean_real = np.nanmean(comparison.real.zeta_hat)
mean_redshift = np.nanmean(comparison.redshift.zeta_hat)

print(f"N_c = {centers.n_centers} centers (shared)")
print(f"v_margin = {centers.v_margin_kms:.1f} km/s = {centers.v_margin_mpc:.2f} h^-1 Mpc")
print(f"n_dropped_v_r_percentile = {centers.n_dropped_v_r_percentile}, "
      f"n_dropped_flip = {centers.n_dropped_flip}")
print(f"shell-averaged zeta_hat_1: real={mean_real:.2f} km/s, redshift={mean_redshift:.2f} km/s, "
      f"ratio={mean_redshift / mean_real:.3f}")
print(f"tracer flux: real-inside-R_sub={tracers.n_real_inside}, "
      f"redshift-inside-R_sub={tracers.n_redshift_inside}")